<a href="https://colab.research.google.com/github/WTeodoro1/Analista-de-dados/blob/Calculadora/otimizacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window


In [6]:
#SparkSession e criaçao do DF Video
spark = SparkSession.builder.getOrCreate()
df_video = spark.read.parquet('/content/videos-preparados.snappy.parquet')
df_comments = spark.read.parquet('/content/videos-comments-tratados.snappy.parquet')
show = df_video.show()
show = df_comments.show()

+--------------------+-----------+------------+----------------+------+--------+---------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+
|               Title|   Video ID|Published At|         Keyword| Likes|Comments|    Views|Interaction|Year|Month|Keyword Index|        Features PCA|     Features Normal|            Features|
+--------------------+-----------+------------+----------------+------+--------+---------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+
|ASMR MUKBANG DOUB...|--ZI0dSbbNU|  2020-04-18|         mukbang|378858|   18860| 17975269|   18372987|2020|    4|         30.0|[0.6985786560867407]|[0.02303716158264...|[378858.0,1.79752...|
|Deadly car bomb d...|--hxd1CrOqg|  2022-08-22|            news|  6379|    4853|   808787|     820019|2022|    8|         37.0|[0.8936407990235931]|[3.87946679100418...|[6379.0,808787.0,...|
|How Biden&#39;s s...|--ixiTypG8g|  2022-08-2

In [7]:
df_video.createOrReplaceTempView("tb_video")
df_comments.createOrReplaceTempView("tb_comments")


In [16]:
join_video_comments = spark.sql("""
    SELECT *
    FROM tb_video v
    JOIN tb_comments c
    ON `v`.`Video ID` = `c`.`Video ID`
""")
show = join_video_comments.show()

+--------------------+-----------+------------+-------+-----+--------+-------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+
|               Title|   Video ID|Published At|Keyword|Likes|Comments|  Views|Interaction|Year|Month|Keyword Index|        Features PCA|     Features Normal|            Features|   Video ID|               Title|Published At|Keyword|Likes|Comments|  Views|Interaction|Year|             Comment|Sentiment|Likes Comment|
+--------------------+-----------+------------+-------+-----+--------+-------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+
|Apple Pay Is Kill...|wAZZ-UWGVHI|  2022-08-23

In [ ]:
#media e variancia
df_video.groupBy("Keyword").agg(
    avg("Views").alias("Avg_Views"),
    variance("Views").alias("Var_Views")
    ).show()

+----------------+--------------------+--------------------+
|         Keyword|           Avg_Views|           Var_Views|
+----------------+--------------------+--------------------+
|computer science|  1191958.7083333333| 2.81219868165102E12|
|            lofi|           4089363.0|1.846209641476677...|
|         finance|   694223.4358974359|3.304483175097042...|
|             cnn|           554240.38|1.563423618468118...|
|           apple|1.0746930452380951E7|4.299927977442589E15|
|            news|   247492.1794871795|1.067512576672564...|
|         mukbang|1.0904772355555555E7|5.586073238973179...|
|       education|  2684432.8333333335|1.833572249339214...|
|       interview|          2966111.86|1.819220996034335E13|
|          crypto|           404608.22|3.513691634369074E12|
|   mathchemistry|  3328125.2666666666|2.491467065256849...|
|            food|          5252406.25|7.326374128154842E13|
|    data science|           544771.98|5.479336525349994...|
|        trolling|      

In [25]:
df_video_repart = spark.read.parquet('/content/videos-preparados.snappy.parquet').repartition(4)
df_comments_repart = spark.read.parquet('/content/videos-comments-tratados.snappy.parquet').repartition(4)

In [20]:
df_video_repart.createOrReplaceTempView("tb_video_repart")
df_comments_repart.createOrReplaceTempView("tb_comments_repart")

In [23]:
join_video_comments_repart = spark.sql("""
    SELECT *
    FROM tb_video_repart v
    JOIN tb_comments_repart c
    ON `v`.`Video ID` = `c`.`Video ID`
""")
show = join_video_comments_repart.show()

+--------------------+-----------+------------+----------------+-------+--------+--------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+----------------+-------+--------+--------+-----------+----+--------------------+---------+-------------+
|               Title|   Video ID|Published At|         Keyword|  Likes|Comments|   Views|Interaction|Year|Month|Keyword Index|        Features PCA|     Features Normal|            Features|   Video ID|               Title|Published At|         Keyword|  Likes|Comments|   Views|Interaction|Year|             Comment|Sentiment|Likes Comment|
+--------------------+-----------+------------+----------------+-------+--------+--------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+----------------+-------+--------+--------+-----------+----+--------------------

In [26]:
df_video_coalesce = spark.read.parquet('/content/videos-preparados.snappy.parquet').coalesce(2)
df_comments_coalesce = spark.read.parquet('/content/videos-comments-tratados.snappy.parquet').coalesce(2)

In [27]:
df_video_coalesce.createOrReplaceTempView("tb_video_coalesce")
df_comments_coalesce.createOrReplaceTempView("tb_comments_coalesce")

In [28]:
join_video_comments_coalesce = spark.sql("""
    SELECT *
    FROM tb_video_coalesce v
    JOIN tb_comments_coalesce c
    ON `v`.`Video ID` = `c`.`Video ID`
""")

In [30]:
#Etapa 6 (EXPLAIN): Serve para visualizar o plano de execução físico e lógico do Spark, mostrando como ele vai processar os dados
print("=== Plano de execução com repartition ===")
join_video_comments_repart.explain()
print("=== Plano de execução com coalesce ===")
join_video_comments_coalesce.explain()


=== Plano de execução com repartition ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [Video ID#542], [Video ID#569], Inner, BuildLeft, false
   :- BroadcastExchange HashedRelationBroadcastMode(List(input[1, string, false]),false), [plan_id=334]
   :  +- Exchange RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=331]
   :     +- Filter isnotnull(Video ID#542)
   :        +- FileScan parquet [Title#541,Video ID#542,Published At#543,Keyword#544,Likes#545,Comments#546,Views#547,Interaction#548,Year#549,Month#550,Keyword Index#551,Features PCA#552,Features Normal#553,Features#554] Batched: true, DataFilters: [isnotnull(Video ID#542)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/videos-preparados.snappy.parquet], PartitionFilters: [], PushedFilters: [IsNotNull(`Video ID`)], ReadSchema: struct<Title:string,Video ID:string,Published At:date,Keyword:string,Likes:int,Comments:int,Views...
   +- Exchange RoundRobinPartitioning(4), RE

In [41]:
#Evitamos leitura de dados que não serão usados, economizando tempo e memória, Selecionamos apenas as colunas necessárias para evitar transporte e armazenamento de dados desnecessário
df_video_opt = spark.read.parquet('/content/videos-preparados.snappy.parquet')
df_comments_opt = spark.read.parquet('/content/videos-comments-tratados.snappy.parquet')

In [34]:
df_video_opt.createOrReplaceTempView("tb_video_opt")
df_comments_opt.createOrReplaceTempView("tb_comments_opt")

In [39]:
join_video_comments_opt = spark.sql("""
    SELECT v.`Video ID`, v.Title, v.Keyword,
           c.Comment, c.Sentiment, c.`Likes Comment`
    FROM tb_video_opt v
    JOIN tb_comments_opt c
   ON `v`.`Video ID` = `c`.`Video ID`
    WHERE v.`Keyword Index` = 22.0
""")
show = join_video_comments_opt.show()

+-----------+--------------------+--------+--------------------+---------+-------------+
|   Video ID|               Title| Keyword|             Comment|Sentiment|Likes Comment|
+-----------+--------------------+--------+--------------------+---------+-------------+
|mQqVCYDD3sA|Kirby’s Dream Buf...|nintendo|Kirby this year r...|        2|         5610|
|mQqVCYDD3sA|Kirby’s Dream Buf...|nintendo|This is really co...|        2|          301|
|mQqVCYDD3sA|Kirby’s Dream Buf...|nintendo|Finally, a game t...|        2|          161|
|mQqVCYDD3sA|Kirby’s Dream Buf...|nintendo|I love that Elfil...|        2|           65|
|mQqVCYDD3sA|Kirby’s Dream Buf...|nintendo|I love the part w...|        2|           68|
|mQqVCYDD3sA|Kirby’s Dream Buf...|nintendo|I love how you ca...|        2|         1846|
|mQqVCYDD3sA|Kirby’s Dream Buf...|nintendo|I've seen art of ...|        2|           73|
|mQqVCYDD3sA|Kirby’s Dream Buf...|nintendo|Yo! This looks so...|     NULL|         NULL|
|mQqVCYDD3sA|Kirby’s 

In [40]:
join_video_comments_opt.write.mode("overwrite").parquet("join-videos-comments-otimizado")
